# Advanced Pufferfish Optimization Algorithm (A-POA)
## Tailored for the CEC 2017 Benchmark Suite

### Background – Standard POA
The **Pufferfish Optimization Algorithm (POA)** is a nature-inspired metaheuristic modelled
on the defensive behaviours of pufferfish. Like many swarm-based optimizers it alternates
between two phases:

| Phase | Biological Analogy | Algorithmic Role |
|-------|--------------------|------------------|
| **Phase 1 – Exploration** | Swimming toward a predator pack | Large jumps guided by elite solutions |
| **Phase 2 – Exploitation** | Inflating / deflating body | Local perturbations around promising regions |

### Why A-POA?
The **CEC 2017** benchmark suite (30 functions, 10-/30-/50-/100-D) introduces:
* **Rotated** search landscapes – axis-aligned updates become inefficient.
* **Shifted** global optima – hard-coded centre biases fail.
* **Composite / hybrid** functions – interleaved multimodal & unimodal basins require
  dynamic exploration-exploitation balance.

This notebook implements **five targeted improvements** over vanilla POA to address these challenges:

| # | Improvement | CEC 2017 Challenge Addressed |
|---|-------------|------------------------------|
| 1 | Coordinated Predator Pack Interaction (CPPI) | Avoids single-elite bias, mitigates local-optima traps |
| 2 | Rotation-Invariant Streamline Update (RISU) | Direction-vector moves instead of per-dimension updates |
| 3 | Density-Adaptive Inflation (Tidal Pressure) | Crowding-aware exploitation radius |
| 4 | Orthogonal Defensive Spine Perturbation | Hyper-local gradient probe for narrow valleys |
| 5 | Buoyancy-Driven Strategy Control | Dynamic phase balancing via success/failure memory |

In [1]:
# ============================================================
# Step 1 – Imports & Setup
# ============================================================
import numpy as np
from scipy.spatial.distance import cdist
import math
from typing import Tuple, Callable

# Reproducibility
np.random.seed(42)

print("All imports successful – environment ready.")

All imports successful – environment ready.


## Step 2 – Global Parameters & State Initialisation

Every agent in the swarm carries three pieces of state:

| Array | Shape | Purpose |
|-------|-------|---------|
| `positions` | `(pop_size, dim)` | Current search-space coordinates |
| `fitness` | `(pop_size,)` | Objective value; initialised to +infinity (unevaluated) |
| `buoyancy_scores` | `(pop_size,)` | Signed counter tracking consecutive successes (+) / failures (-) |

**Buoyancy** is a novel addition: it lets the algorithm *remember* per-agent history
and adapt strategy accordingly (see Step 7).

In [2]:
# ============================================================
# Step 2 – Population Initialisation
# ============================================================

def initialize_population(
    pop_size: int,
    dim: int,
    lower_bound: float,
    upper_bound: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Create a uniformly random initial swarm.

    Parameters
    ----------
    pop_size : int
        Number of agents in the population.
    dim : int
        Dimensionality of the search space.
    lower_bound : float
        Lower bound of the search space (same for every dimension).
    upper_bound : float
        Upper bound of the search space (same for every dimension).

    Returns
    -------
    positions : np.ndarray, shape (pop_size, dim)
        Random initial positions within [lower_bound, upper_bound].
    fitness : np.ndarray, shape (pop_size,)
        Initialised to +inf (not yet evaluated).
    buoyancy_scores : np.ndarray, shape (pop_size,)
        Initialised to 0 (neutral history).
    """
    # Uniform random initialisation across the feasible region
    positions: np.ndarray = np.random.uniform(
        low=lower_bound, high=upper_bound, size=(pop_size, dim)
    )

    # Fitness unknown -> +inf acts as "worst possible" sentinel
    fitness: np.ndarray = np.full(pop_size, np.inf)

    # Buoyancy starts neutral for every agent
    buoyancy_scores: np.ndarray = np.zeros(pop_size)

    return positions, fitness, buoyancy_scores


# --- Quick sanity check ---
_pos, _fit, _buoy = initialize_population(pop_size=5, dim=3, lower_bound=-100, upper_bound=100)
print(f"positions shape : {_pos.shape}")   # (5, 3)
print(f"fitness shape   : {_fit.shape}")   # (5,)
print(f"buoyancy shape  : {_buoy.shape}")  # (5,)
print(f"fitness[0]      : {_fit[0]}")      # inf
print(f"buoyancy[0]     : {_buoy[0]}")     # 0.0

positions shape : (5, 3)
fitness shape   : (5,)
buoyancy shape  : (5,)
fitness[0]      : inf
buoyancy[0]     : 0.0


## Step 3 – Improvement 1: Coordinated Predator Pack Interaction (CPPI)

### Problem with Standard POA
In vanilla POA, the exploration phase moves each agent toward a **single randomly chosen
elite**. This creates a strong attractor that can trap the entire swarm in one basin –
catastrophic on CEC 2017's multimodal composites (F21–F30).

### Solution – CPPI
Instead of a single elite, we compute a **weighted spatial centroid** of the top-$K$
individuals:

$$\mathbf{x}_{\text{cppi}} = \frac{\sum_{k=1}^{K} w_k \, \mathbf{x}_k}{\sum_{k=1}^{K} w_k}$$

where the weights are **inversely proportional** to fitness (lower is better):

$$w_k = \frac{1}{f(\mathbf{x}_k) + \varepsilon}, \quad \varepsilon = 10^{-8}$$

This ensures the centroid is pulled more strongly toward the *best* of the elites while
still being influenced by alternative basins represented by the other top-$K$ members.

In [3]:
# ============================================================
# Step 3 – Coordinated Predator Pack Interaction (CPPI)
# ============================================================

def get_cppi_target(
    positions: np.ndarray,
    fitness: np.ndarray,
    K: int = 3,
) -> np.ndarray:
    """Compute the weighted centroid of the top-K fittest individuals.

    Parameters
    ----------
    positions : np.ndarray, shape (pop_size, dim)
    fitness : np.ndarray, shape (pop_size,)
    K : int
        Number of elite individuals to consider (default 3).

    Returns
    -------
    cppi_target : np.ndarray, shape (dim,)
        Weighted centroid of the top-K individuals.
    """
    # Ensure K does not exceed population size
    K = min(K, len(fitness))

    # Indices sorted by ascending fitness (best first)
    sorted_indices: np.ndarray = np.argsort(fitness)
    top_k_indices: np.ndarray = sorted_indices[:K]

    # Select elite positions and their fitness values
    elite_positions: np.ndarray = positions[top_k_indices]   # (K, dim)
    elite_fitness: np.ndarray = fitness[top_k_indices]        # (K,)

    # Inverse-fitness weights (epsilon avoids division by zero)
    epsilon: float = 1e-8
    weights: np.ndarray = 1.0 / (elite_fitness + epsilon)     # (K,)

    # Weighted centroid
    cppi_target: np.ndarray = np.average(elite_positions, axis=0, weights=weights)

    return cppi_target


# --- Quick demo ---
_demo_pos = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]], dtype=float)
_demo_fit = np.array([10.0, 2.0, 5.0, 1.0, 8.0])
_target = get_cppi_target(_demo_pos, _demo_fit, K=3)
print(f"CPPI target (K=3): {_target}")

CPPI target (K=3): [5.58823529 6.58823529]


## Step 4 – Improvement 2: Rotation-Invariant Streamline Update (RISU)

### Problem
CEC 2017 functions are applied **after** rotating the search space with an orthogonal
matrix $\mathbf{M}$. Standard POA perturbs each dimension independently, which is
inherently **axis-aligned** – the perturbation ellipsoid's principal axes are always
parallel to the coordinate axes, making it blind to the rotated landscape's true
structure.

### Solution – RISU
Instead of per-dimension random steps, RISU moves along the **exact direction vector**
from the current position to the target:

$$\mathbf{V} = \mathbf{x}_{\text{target}} - I \cdot \mathbf{x}_{\text{current}}, \quad I \in \{1, 2\}$$

The factor $I$ (randomly chosen as 1 or 2) controls whether the agent moves toward or
overshoots the target – this adds beneficial diversity. The update is:

$$\mathbf{x}_{\text{new}} = \mathbf{x}_{\text{current}} + r \cdot \mathbf{V} + \mathcal{N}(0, 0.01)$$

where $r \sim U(0,1)$ is a **single scalar** (not a vector), preserving the direction,
and $\mathcal{N}(0, 0.01)$ is a small **isotropic** Gaussian noise that prevents
co-linear collapse.

In [4]:
# ============================================================
# Step 4 – Rotation-Invariant Streamline Update (RISU)
# ============================================================

def risu_update(
    current_pos: np.ndarray,
    target_pos: np.ndarray,
    dim: int,
) -> np.ndarray:
    """Compute a rotation-invariant position update toward a target.

    Parameters
    ----------
    current_pos : np.ndarray, shape (dim,)
        Current agent position.
    target_pos : np.ndarray, shape (dim,)
        Target position (e.g. CPPI centroid).
    dim : int
        Dimensionality of the search space.

    Returns
    -------
    new_pos : np.ndarray, shape (dim,)
        Updated agent position.
    """
    # Random multiplier I in {1, 2} for the current position
    I: int = np.random.choice([1, 2])

    # Direction vector from (I * current) to the target
    direction: np.ndarray = target_pos - (I * current_pos)   # (dim,)

    # Single scalar step size preserves the direction
    r: float = np.random.uniform(0.0, 1.0)

    # Isotropic Gaussian noise for diversity (small sigma = 0.01)
    noise: np.ndarray = np.random.normal(loc=0.0, scale=0.01, size=dim)

    # Updated position
    new_pos: np.ndarray = current_pos + r * direction + noise

    return new_pos


# --- Quick demo ---
_cur = np.array([1.0, 2.0, 3.0])
_tgt = np.array([5.0, 5.0, 5.0])
_new = risu_update(_cur, _tgt, dim=3)
print(f"Current  : {_cur}")
print(f"Target   : {_tgt}")
print(f"RISU new : {_new}")

Current  : [1. 2. 3.]
Target   : [5. 5. 5.]
RISU new : [3.47014826 3.8481666  4.22964485]


## Step 5 – Improvement 3: Density-Adaptive Inflation (Tidal Pressure)

### Problem
Standard POA shrinks the exploitation (Phase 2) perturbation radius **linearly** with
iteration count. This ignores the actual spatial distribution of agents: in a crowded
region the step should be **larger** to escape, while an isolated agent should keep a
small, precise step.

### Solution – Tidal Pressure
We compute a **Density Factor** ($DF$) for each agent based on its mean distance to all
other agents:

$$\bar{d}_i = \frac{1}{N-1} \sum_{j \neq i} \|\mathbf{x}_i - \mathbf{x}_j\|_2$$

$$DF_i = 1 - \frac{\bar{d}_i - \bar{d}_{\min}}{\bar{d}_{\max} - \bar{d}_{\min} + \varepsilon}
\;\in [0,\,1]$$

A $DF$ close to **1** means high crowding; close to **0** means isolation. The adaptive
step size is:

$$s_i = s_{\text{base}} \cdot (1 + 0.5 \cdot DF_i)$$

This gives crowded agents up to **50 %** extra reach.

In [5]:
# ============================================================
# Step 5 – Density-Adaptive Inflation (Tidal Pressure)
# ============================================================

def calculate_tidal_pressure_step(
    current_idx: int,
    positions: np.ndarray,
    base_step_size: float,
) -> float:
    """Return an adaptive step size based on local crowding.

    Parameters
    ----------
    current_idx : int
        Index of the agent for which we compute the step.
    positions : np.ndarray, shape (pop_size, dim)
        All agent positions.
    base_step_size : float
        The unadapted (base) step size.

    Returns
    -------
    adaptive_step : float
        Step size scaled by the density factor.
    """
    pop_size: int = positions.shape[0]

    # Pairwise Euclidean distances from current agent to every other agent
    current_pos: np.ndarray = positions[current_idx].reshape(1, -1)       # (1, dim)
    distances: np.ndarray = cdist(current_pos, positions, metric="euclidean").flatten()

    # Exclude self-distance (which is 0)
    other_distances: np.ndarray = np.delete(distances, current_idx)

    # Mean distance for this agent
    mean_dist: float = float(np.mean(other_distances))

    # Compute mean distances for ALL agents (needed for min/max normalisation)
    all_mean_dists: np.ndarray = np.zeros(pop_size)
    full_dist_matrix: np.ndarray = cdist(positions, positions, metric="euclidean")
    for i in range(pop_size):
        row = np.delete(full_dist_matrix[i], i)
        all_mean_dists[i] = np.mean(row)

    d_min: float = float(np.min(all_mean_dists))
    d_max: float = float(np.max(all_mean_dists))

    # Density Factor (1 = crowded, 0 = isolated)
    epsilon: float = 1e-8
    density_factor: float = 1.0 - (mean_dist - d_min) / (d_max - d_min + epsilon)
    density_factor = float(np.clip(density_factor, 0.0, 1.0))

    # Adaptive step: crowded agents get up to 50 % more reach
    adaptive_step: float = base_step_size * (1.0 + 0.5 * density_factor)

    return adaptive_step


# --- Quick demo ---
_demo_positions = np.array([[0, 0], [1, 1], [1.1, 1.1], [10, 10], [10.1, 10.1]], dtype=float)
for idx in range(5):
    step = calculate_tidal_pressure_step(idx, _demo_positions, base_step_size=1.0)
    print(f"Agent {idx}: adaptive step = {step:.4f}")

Agent 0: adaptive step = 1.3315
Agent 1: adaptive step = 1.4946
Agent 2: adaptive step = 1.5000
Agent 3: adaptive step = 1.0163
Agent 4: adaptive step = 1.0000


## Step 6 – Improvement 4: Orthogonal Defensive Spine Perturbation

### Motivation
CEC 2017's composite functions (F21–F30) often feature **narrow curved valleys**. Even
after reaching the right basin, progress stalls because the optimizer lacks a fine-grained
directional probe.

### Solution
After a successful iteration the algorithm performs a **greedy coordinate-wise micro-
perturbation** (inspired by coordinate descent):

```
for each dimension j:
    try  x_j + epsilon   ->  if f improves, accept & return
    try  x_j - epsilon   ->  if f improves, accept & return
```

The perturbation uses a tiny $\varepsilon = 10^{-4}$, making it essentially a **discrete
one-sided gradient check** along each axis. Because it returns at the *first* improvement,
it is cheap ($\mathcal{O}(\text{dim})$ in expectation).

In [6]:
# ============================================================
# Step 6 – Orthogonal Defensive Spine Perturbation
# ============================================================

def orthogonal_spine_perturbation(
    position: np.ndarray,
    current_fitness: float,
    obj_function: Callable[[np.ndarray], float],
    lower_bound: float,
    upper_bound: float,
    epsilon: float = 1e-4,
) -> Tuple[np.ndarray, float]:
    """Greedy per-dimension micro-perturbation.

    Parameters
    ----------
    position : np.ndarray, shape (dim,)
        Current agent position (will be copied, not mutated).
    current_fitness : float
        f(position).
    obj_function : callable
        Objective function  f : R^dim -> R.
    lower_bound, upper_bound : float
        Search-space bounds.
    epsilon : float
        Perturbation magnitude (default 1e-4).

    Returns
    -------
    best_pos : np.ndarray, shape (dim,)
        Possibly improved position.
    best_fit : float
        Fitness at best_pos.
    """
    best_pos: np.ndarray = position.copy()
    best_fit: float = current_fitness
    dim: int = len(position)

    for j in range(dim):
        # --- Try +epsilon along dimension j ---
        trial_pos: np.ndarray = best_pos.copy()
        trial_pos[j] += epsilon
        trial_pos = np.clip(trial_pos, lower_bound, upper_bound)
        trial_fit: float = obj_function(trial_pos)

        if trial_fit < best_fit:
            best_pos = trial_pos
            best_fit = trial_fit
            return best_pos, best_fit  # greedy early exit

        # --- Try -epsilon along dimension j ---
        trial_pos = best_pos.copy()
        trial_pos[j] -= epsilon
        trial_pos = np.clip(trial_pos, lower_bound, upper_bound)
        trial_fit = obj_function(trial_pos)

        if trial_fit < best_fit:
            best_pos = trial_pos
            best_fit = trial_fit
            return best_pos, best_fit  # greedy early exit

    # No improvement found in any dimension
    return best_pos, best_fit


# --- Quick demo on a simple sphere ---
_sphere = lambda x: float(np.sum(x**2))
_p = np.array([0.5, 0.5, 0.5])
_f = _sphere(_p)
_p2, _f2 = orthogonal_spine_perturbation(_p, _f, _sphere, -100, 100)
print(f"Before: f={_f:.8f}  pos={_p}")
print(f"After : f={_f2:.8f}  pos={_p2}")

Before: f=0.75000000  pos=[0.5 0.5 0.5]
After : f=0.74990001  pos=[0.4999 0.5    0.5   ]


## Step 7 – Improvement 5: Buoyancy-Driven Strategy Control & Chaotic Drift

### Per-Agent Memory – Buoyancy
Each agent maintains a signed integer **buoyancy score**:
* On a **successful** iteration (fitness improved): `buoyancy += 1`
* On a **failed** iteration: `buoyancy -= 1`

This counter drives a three-tier strategy selector in the main loop:

| Buoyancy Range | Interpretation | Action |
|----------------|----------------|--------|
| `buoyancy < -3` | Deeply stagnant | **Chaotic Drift** – teleport via logistic map |
| `buoyancy <= 0` | Mildly stagnant / neutral | **Phase 1** (Exploration via RISU) |
| `buoyancy > 0` | Improving | **Phase 2** (Exploitation) + Spine Perturbation |

### Chaotic Drift via the Logistic Map
When an agent is deeply stagnant we use the **logistic map** to generate a deterministic
but chaotic sequence:

$$z_{n+1} = 4 \, z_n \, (1 - z_n), \quad z_0 \in (0, 1)$$

A *dim*-length chaotic vector $\mathbf{c}$ is produced and then linearly mapped to the
search-space bounds:

$$x_j^{\text{new}} = L + c_j \cdot (U - L)$$

This **aggressively ejects** the agent from its local minimum into a quasi-random yet
deterministic new region.

In [7]:
# ============================================================
# Step 7 – Chaotic Drift (Logistic Map Teleportation)
# ============================================================

def chaotic_drift(
    position: np.ndarray,
    lower_bound: float,
    upper_bound: float,
) -> np.ndarray:
    """Teleport an agent using a logistic-map chaotic sequence.

    Parameters
    ----------
    position : np.ndarray, shape (dim,)
        Current position (used only to infer dimensionality).
    lower_bound, upper_bound : float
        Search-space bounds.

    Returns
    -------
    new_pos : np.ndarray, shape (dim,)
        New position generated via chaotic mapping.
    """
    dim: int = len(position)

    # Seed the logistic map with a random value in (0, 1)
    z: float = np.random.uniform(0.01, 0.99)

    chaotic_vector: np.ndarray = np.zeros(dim)
    for j in range(dim):
        z = 4.0 * z * (1.0 - z)          # logistic map iteration
        chaotic_vector[j] = z

    # Scale chaotic vector [0, 1] -> [lower_bound, upper_bound]
    new_pos: np.ndarray = lower_bound + chaotic_vector * (upper_bound - lower_bound)

    return new_pos


# --- Quick demo ---
_chaotic_pos = chaotic_drift(np.zeros(5), -100, 100)
print(f"Chaotic drift result (5-D): {_chaotic_pos}")

Chaotic drift result (5-D): [-72.44503815  -4.96567104  99.50684222 -98.03223298 -92.20637406]


## Step 8 – The Main Loop Assembly

All five improvements are now integrated into a single optimisation loop:

```
for each iteration:
    1.  cppi_target = weighted centroid of top-K (CPPI)
    2.  for each agent i:
        a.  if buoyancy[i] < -3   ->  Chaotic Drift, reset buoyancy
        b.  if buoyancy[i] <=  0  ->  Phase 1 (RISU exploration toward cppi_target)
        c.  Phase 2 exploitation with Tidal-Pressure-adapted step size
        d.  If fitness improved   ->  buoyancy += 1, apply Spine Perturbation
            Else                  ->  buoyancy -= 1
    3.  Update global best
```

The function returns the best position and fitness found across all iterations.

In [8]:
# ============================================================
# Step 8 – Main A-POA Loop
# ============================================================

def run_apoa(
    obj_function: Callable[[np.ndarray], float],
    dim: int,
    pop_size: int = 30,
    max_iter: int = 500,
    lower_bound: float = -100.0,
    upper_bound: float = 100.0,
    K: int = 3,
) -> Tuple[np.ndarray, float]:
    """Run the Advanced Pufferfish Optimization Algorithm.

    Parameters
    ----------
    obj_function : callable
        Objective function  f : R^dim -> R  (minimisation).
    dim : int
        Problem dimensionality.
    pop_size : int
        Number of search agents.
    max_iter : int
        Maximum iterations.
    lower_bound, upper_bound : float
        Search-space bounds (symmetric across all dimensions).
    K : int
        Number of elites for CPPI (default 3).

    Returns
    -------
    global_best_pos : np.ndarray, shape (dim,)
    global_best_fit : float
    """
    # -- Initialisation --
    positions, fitness, buoyancy = initialize_population(
        pop_size, dim, lower_bound, upper_bound
    )

    # Evaluate initial population
    for i in range(pop_size):
        fitness[i] = obj_function(positions[i])

    # Track global best
    best_idx: int = int(np.argmin(fitness))
    global_best_pos: np.ndarray = positions[best_idx].copy()
    global_best_fit: float = fitness[best_idx]

    # Convergence curve (optional, handy for plotting)
    convergence: list = []

    # -- Main Loop --
    for t in range(max_iter):
        # 1. Compute CPPI target (weighted centroid of top-K)
        cppi_target: np.ndarray = get_cppi_target(positions, fitness, K=K)

        # Base step size decays linearly from 1 to 0
        base_step: float = 1.0 - (t / max_iter)

        for i in range(pop_size):
            old_fit: float = fitness[i]

            # -- Buoyancy-driven strategy selection --
            if buoyancy[i] < -3:
                # Deeply stagnant: chaotic drift (teleport)
                positions[i] = chaotic_drift(positions[i], lower_bound, upper_bound)
                positions[i] = np.clip(positions[i], lower_bound, upper_bound)
                fitness[i] = obj_function(positions[i])
                buoyancy[i] = 0  # reset buoyancy after drift
                old_fit = fitness[i]  # update reference for later comparison

            if buoyancy[i] <= 0:
                # Neutral / mildly stagnant: Phase 1 Exploration (RISU)
                new_pos: np.ndarray = risu_update(positions[i], cppi_target, dim)
                new_pos = np.clip(new_pos, lower_bound, upper_bound)
                new_fit: float = obj_function(new_pos)
                # Accept if improved (greedy)
                if new_fit < fitness[i]:
                    positions[i] = new_pos
                    fitness[i] = new_fit

            # -- Phase 2 Exploitation (always attempted) --
            adaptive_step: float = calculate_tidal_pressure_step(
                i, positions, base_step
            )

            # Random perturbation within adaptive radius
            perturbation: np.ndarray = np.random.uniform(
                -adaptive_step, adaptive_step, size=dim
            )
            trial_pos: np.ndarray = positions[i] + perturbation
            trial_pos = np.clip(trial_pos, lower_bound, upper_bound)
            trial_fit: float = obj_function(trial_pos)

            if trial_fit < fitness[i]:
                positions[i] = trial_pos
                fitness[i] = trial_fit

            # -- Buoyancy update --
            if fitness[i] < old_fit:
                buoyancy[i] += 1
                # Success: try orthogonal spine perturbation
                positions[i], fitness[i] = orthogonal_spine_perturbation(
                    positions[i], fitness[i], obj_function,
                    lower_bound, upper_bound,
                )
            else:
                buoyancy[i] -= 1

            # -- Global-best update --
            if fitness[i] < global_best_fit:
                global_best_fit = fitness[i]
                global_best_pos = positions[i].copy()

        convergence.append(global_best_fit)

        # Optional progress print every 100 iterations
        if (t + 1) % 100 == 0 or t == 0:
            print(f"Iter {t+1:>4d}/{max_iter}  |  Best fitness: {global_best_fit:.6e}")

    print(f"")
    print(f"{'=' * 50}")
    print(f"Optimisation complete.")
    print(f"Best fitness : {global_best_fit:.6e}")
    print(f"Best position: {global_best_pos[:5]}{'...' if dim > 5 else ''}")
    print(f"{'=' * 50}")

    return global_best_pos, global_best_fit

### Test Run – 10-D Sphere Function

As a sanity check we run A-POA on the classic **Sphere function**:

$$f(\mathbf{x}) = \sum_{i=1}^{D} x_i^2, \quad \mathbf{x}^* = \mathbf{0}, \; f^* = 0$$

Expected outcome: the best fitness should converge very close to **0**.

In [9]:
# ============================================================
# Test – 10-D Sphere Function
# ============================================================

def sphere(x: np.ndarray) -> float:
    """Classic sphere function: f(x) = sum(x_i^2)."""
    return float(np.sum(x ** 2))


best_pos, best_fit = run_apoa(
    obj_function=sphere,
    dim=10,
    pop_size=30,
    max_iter=500,
    lower_bound=-100.0,
    upper_bound=100.0,
)

print(f"\nFinal best fitness on 10-D Sphere: {best_fit:.10e}")

Iter    1/500  |  Best fitness: 1.212131e+03


Iter  100/500  |  Best fitness: 2.239684e-04


Iter  200/500  |  Best fitness: 2.239684e-04


Iter  300/500  |  Best fitness: 2.239684e-04


Iter  400/500  |  Best fitness: 2.239684e-04


Iter  500/500  |  Best fitness: 1.450414e-04

Optimisation complete.
Best fitness : 1.450414e-04
Best position: [-0.0003628  -0.00330223 -0.00106063  0.00095294 -0.00152829]...

Final best fitness on 10-D Sphere: 1.4504144793e-04
